# SimpleNews S0–S5


This version is configured to run **top-to-bottom** without manual test cells.

Notes:
- The **S5 helper cell only defines functions** and prints `S5 helper functions loaded.`
- The **live progress bars appear in Step 9**, when the notebook actually runs **S4** and **S5**
- Use this notebook with the corrected existing `src/` folder beside the notebook


## 0. Setup and existing shared-module import

This cell:
1. uses the existing local `src/` package already present next to the notebook,
2. clears stale imports / `__pycache__`,
3. imports the shared modules, and
4. verifies the expected selector / simplifier signatures.

No zip extraction or file copying is done here.


In [1]:

from pathlib import Path
import sys
import os
import shutil
import importlib
import inspect

ROOT = Path('.').resolve()
SRC = ROOT / "src"

module_files = [
    "__init__.py",
    "config.py",
    "dataset_bootstrap.py",
    "evaluation.py",
    "io_utils.py",
    "pipeline.py",
    "preprocess.py",
    "selectors.py",
    "simplify.py",
]

if not SRC.exists():
    raise FileNotFoundError(
        f"Expected an existing src/ folder at {SRC.resolve()} but it was not found. "
        "Place the unzipped improved src folder next to this notebook and rerun."
    )

missing = [name for name in module_files if not (SRC / name).exists()]
if missing:
    raise FileNotFoundError(
        f"The existing src/ folder is missing required files: {missing}.\n"
        f"Checked folder: {SRC.resolve()}"
    )

print(f"Using existing src/ at: {SRC.resolve()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Clear stale bytecode + stale imports so old versions do not linger.
shutil.rmtree(SRC / "__pycache__", ignore_errors=True)

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

import src
import src.config
import src.dataset_bootstrap
import src.evaluation
import src.io_utils
import src.preprocess
import src.selectors
import src.simplify
import src.pipeline

importlib.reload(src)
importlib.reload(src.config)
importlib.reload(src.dataset_bootstrap)
importlib.reload(src.evaluation)
importlib.reload(src.io_utils)
importlib.reload(src.preprocess)
importlib.reload(src.selectors)
importlib.reload(src.simplify)
importlib.reload(src.pipeline)

from src.config import Paths, DEFAULT_CONFIG
from src.dataset_bootstrap import ensure_dataset
from src.io_utils import ensure_dirs, load_csv, standardize_dataframe
from src.pipeline import generate_system_outputs
from src.evaluation import evaluate_output
from src.selectors import textrank_select, coverage_aware_select
from src.simplify import conservative_lexical_simplify
from src.preprocess import token_frequency_rank, normalize_whitespace

print("textrank_select signature:", inspect.signature(textrank_select))
print("coverage_aware_select signature:", inspect.signature(coverage_aware_select))
print("conservative_lexical_simplify signature:", inspect.signature(conservative_lexical_simplify))
print("Existing src/ package ready.")


# Extra sanity checks so mixed old/new module bindings fail early.
import src.selectors as _src_selectors
import src.pipeline as _src_pipeline

print("src.selectors file:", Path(_src_selectors.__file__).resolve())
print("src.pipeline file:", Path(_src_pipeline.__file__).resolve())

sel_sig = inspect.signature(_src_selectors.textrank_select)
pipe_sel_sig = inspect.signature(_src_pipeline.textrank_select)
cov_sig = inspect.signature(_src_selectors.coverage_aware_select)
pipe_cov_sig = inspect.signature(_src_pipeline.coverage_aware_select)

assert "similarity_threshold" in sel_sig.parameters, (
    f"src.selectors.textrank_select is stale: {sel_sig}"
)
assert "similarity_threshold" in pipe_sel_sig.parameters, (
    f"src.pipeline.textrank_select binding is stale: {pipe_sel_sig}"
)
assert "similarity_threshold" in cov_sig.parameters, (
    f"src.selectors.coverage_aware_select is stale: {cov_sig}"
)
assert "require_number_coverage" in cov_sig.parameters and "require_date_coverage" in cov_sig.parameters, (
    f"src.selectors.coverage_aware_select missing coverage args: {cov_sig}"
)
assert "require_number_coverage" in pipe_cov_sig.parameters and "require_date_coverage" in pipe_cov_sig.parameters, (
    f"src.pipeline.coverage_aware_select binding is stale: {pipe_cov_sig}"
)
assert _src_pipeline.textrank_select is _src_selectors.textrank_select, (
    "src.pipeline still points to an older textrank_select binding. Restart kernel and rerun from the top."
)
assert _src_pipeline.coverage_aware_select is _src_selectors.coverage_aware_select, (
    "src.pipeline still points to an older coverage_aware_select binding. Restart kernel and rerun from the top."
)

print("Selector/pipeline bindings are synced.")


Using existing src/ at: /Users/gm/Desktop/4/src
textrank_select signature: (article: 'str', max_sentences: 'int' = 3, similarity_threshold: 'float' = 0.05) -> 'str'
coverage_aware_select signature: (article: 'str', title: 'str' = '', max_sentences: 'int' = 3, top_entity_k: 'int' = 6, similarity_threshold: 'float' = 0.05, require_number_coverage: 'bool' = True, require_date_coverage: 'bool' = True) -> 'str'
conservative_lexical_simplify signature: (text: 'str', frequency_rank: 'Dict[str, int]', max_word_length: 'int' = 8, min_word_frequency_rank: 'int' = 2000, max_replacements_per_doc: 'int' = 10, protected_words: 'List[str] | None' = None, use_glossary_fallback: 'bool' = True, min_replacement_confidence: 'float' = 0.7, max_glossary_items: 'int' = 8, use_structural_simplification: 'bool' = True) -> 'Tuple[str, List[Dict[str, str]], List[str]]'
Existing src/ package ready.
src.selectors file: /Users/gm/Desktop/4/src/selectors.py
src.pipeline file: /Users/gm/Desktop/4/src/pipeline.py
Sele

## 1. Install / import extra packages for S4 / S5

In [2]:
import sys, subprocess, importlib.util

REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

In [3]:
import gc
import math
import copy
import warnings
import time
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
import torch
from IPython.display import HTML, display
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

# Silence repetitive generation/config warnings that otherwise flood notebook output
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=r".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", message=r".*min_new_tokens.*min_length.*")
warnings.filterwarnings("ignore", message=r".*forced_bos_token_id.*")
warnings.filterwarnings("ignore", message=r".*tie shared.weight to lm_head.weight.*")

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

def runtime_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = runtime_device()
print("Selected device:", DEVICE)

class NotebookProgress:
    def __init__(self, total: int, desc: str, enabled: bool = True, update_every: int = 5):
        self.total = max(int(total), 1)
        self.desc = desc
        self.enabled = enabled
        self.update_every = max(1, int(update_every))
        self.start = time.time()
        self.last_doc = None
        self.handle = None
        if self.enabled:
            self.handle = display(self._render(0), display_id=True)

    def _render(self, n: int):
        pct = min(100.0, 100.0 * n / self.total)
        elapsed = time.time() - self.start
        rate = n / elapsed if elapsed > 0 else 0.0
        eta = (self.total - n) / rate if rate > 0 else 0.0
        elapsed_txt = f"{elapsed/60:.1f} min"
        eta_txt = "--" if rate <= 0 else f"{eta/60:.1f} min"
        last_txt = "" if self.last_doc is None else f" | last_doc={self.last_doc}"
        return HTML(f'''
        <div style="font-family:Arial,sans-serif; margin:8px 0;">
          <div style="margin-bottom:4px;"><b>{self.desc}</b>: {n}/{self.total} ({pct:.1f}%) | elapsed {elapsed_txt} | ETA {eta_txt}{last_txt}</div>
          <div style="width:100%; background:#e9ecef; border-radius:8px; overflow:hidden; height:14px;">
            <div style="width:{pct:.1f}%; background:#2f80ed; height:14px;"></div>
          </div>
        </div>
        ''')

    def update(self, n: int, last_doc: Optional[str] = None, force: bool = False):
        if not self.enabled or self.handle is None:
            return
        if last_doc is not None:
            self.last_doc = last_doc
        if force or n == self.total or n % self.update_every == 0:
            self.handle.update(self._render(n))

    def close(self, last_doc: Optional[str] = None):
        self.update(self.total, last_doc=last_doc, force=True)

def make_progress_bar(total: int, desc: str, enabled: bool = True, update_every: int = 5):
    if not enabled:
        return None
    return NotebookProgress(total=total, desc=desc, enabled=True, update_every=update_every)


Selected device: mps


In [4]:

# Hugging Face authentication (safe notebook-side setup)
# Reads HF_TOKEN from:
#   1) environment variable HF_TOKEN
#   2) optional local text file .hf_token or hf_token in the project root

from pathlib import Path
import os

HF_TOKEN = (os.environ.get("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    for token_file in [ROOT / ".hf_token", ROOT / "hf_token"]:
        if token_file.exists():
            HF_TOKEN = token_file.read_text(encoding="utf-8").strip()
            break

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF authentication configured from environment/local token file.")
    except Exception as e:
        print("HF token found, but explicit login helper was not available or failed.")
        print("Continuing with HF_TOKEN environment variable only.")
        print("Reason:", repr(e))
else:
    print("No HF_TOKEN found. Running unauthenticated (public resources still work, but with lower rate limits).")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF authentication configured from environment/local token file.


## 2. Configuration

In [5]:

paths = Paths(
    root=ROOT,
    data_dir=ROOT / "data",
    output_dir=ROOT / "outputs",
    figure_dir=ROOT / "outputs/figures",
    table_dir=ROOT / "outputs/tables",
    sample_dir=ROOT / "outputs/samples",
    checkpoint_dir=ROOT / "outputs/checkpoints",
)
ensure_dirs(paths.data_dir, paths.output_dir, paths.figure_dir, paths.table_dir, paths.sample_dir, paths.checkpoint_dir)

# Stronger defaults for the improved shared pipeline.
DEFAULT_CONFIG.selector.max_sentences = 3
DEFAULT_CONFIG.selector.similarity_threshold = 0.08
DEFAULT_CONFIG.selector.top_entity_k = 8
DEFAULT_CONFIG.selector.require_number_coverage = True
DEFAULT_CONFIG.selector.require_date_coverage = True

DEFAULT_CONFIG.simplifier.enabled = True
DEFAULT_CONFIG.simplifier.max_word_length = 8
DEFAULT_CONFIG.simplifier.min_word_frequency_rank = 2000
DEFAULT_CONFIG.simplifier.max_replacements_per_doc = 10
DEFAULT_CONFIG.simplifier.min_replacement_confidence = 0.70
DEFAULT_CONFIG.simplifier.max_glossary_items = 8
DEFAULT_CONFIG.simplifier.use_glossary_fallback = True
DEFAULT_CONFIG.simplifier.use_structural_simplification = True

# Dataset split to run on for the final benchmark.
RUN_SPLIT = "test"   # final benchmark split after tuning/fine-tuning
RUN_LIMIT = None           # set to e.g. 100 for debugging, None for full split

# Optional validation tuning sweeps (run BEFORE the full benchmark if you want the tuned settings)
RUN_S3_SWEEP = True
RUN_S4_SELECTOR_SWEEP = True
SWEEP_LIMIT = 300

# Optional S4 fine-tuning workflow (run after sweeps, before the final benchmark)
ENABLE_S4_FINETUNING = True
USE_FINETUNED_S4 = True
S4_FINETUNE_TRAIN_LIMIT = 12000
S4_FINETUNE_VAL_LIMIT = 1000
S4_FINETUNE_EPOCHS = 1
S4_FINETUNE_BATCH_SIZE = 1
S4_FINETUNE_MODEL_NAME = "google/flan-t5-base"

# S5 is kept as an inference-only baseline here. If you later want S5 fine-tuning, do it in a separate notebook.
# LLM systems
ENABLE_LLM_SYSTEMS = True

# Model choices
S4_MODEL_NAME = "google/flan-t5-large"        # stronger controlled rewrite (slower)
S5_MODEL_NAME = "facebook/bart-large-cnn"     # full-article summarization baseline

# Generation lengths / budgets
S4_MAX_NEW_TOKENS = 160
S5_MAX_NEW_TOKENS = 144
S5_MIN_NEW_TOKENS = 64
S5_CHUNK_WORD_BUDGET = 420

# Tuned S4 guard defaults (can be overwritten by sweep results)
S4_MIN_ENTITY_F1 = 0.70
S4_MIN_NUMBER_F1 = 0.82
S4_MIN_DATE_F1 = 0.82
S4_MAX_ENTITY_UNSUPPORTED = 0.15
S4_MAX_NUMBER_UNSUPPORTED = 0.10
S4_MAX_DATE_UNSUPPORTED = 0.10
S4_MIN_WORDS = 50
S4_MAX_WORDS = 110

# Separate selector settings used for the S4 scaffold.
S4_SELECTOR_CONFIG = {
    "max_sentences": 4,
    "top_entity_k": 8,
    "similarity_threshold": 0.08,
    "require_number_coverage": True,
    "require_date_coverage": True,
}

RUN_TAG = f"{RUN_SPLIT}_s0_s5_v25_no_toggle"

print("RUN_SPLIT =", RUN_SPLIT)
print("RUN_LIMIT =", RUN_LIMIT)
print("ENABLE_LLM_SYSTEMS =", ENABLE_LLM_SYSTEMS)
print("RUN_S3_SWEEP =", RUN_S3_SWEEP)
print("RUN_S4_SELECTOR_SWEEP =", RUN_S4_SELECTOR_SWEEP)
print("ENABLE_S4_FINETUNING =", ENABLE_S4_FINETUNING)
print("USE_FINETUNED_S4 =", USE_FINETUNED_S4)
print("S4_MODEL_NAME =", S4_MODEL_NAME)
print("S5_MODEL_NAME =", S5_MODEL_NAME)
print("DEFAULT_CONFIG =", DEFAULT_CONFIG)
print("S4_SELECTOR_CONFIG =", S4_SELECTOR_CONFIG)


RUN_SPLIT = test
RUN_LIMIT = None
ENABLE_LLM_SYSTEMS = True
RUN_S3_SWEEP = True
RUN_S4_SELECTOR_SWEEP = True
ENABLE_S4_FINETUNING = True
USE_FINETUNED_S4 = True
S4_MODEL_NAME = google/flan-t5-large
S5_MODEL_NAME = facebook/bart-large-cnn
DEFAULT_CONFIG = ExperimentConfig(random_seed=42, sample_size=200, systems=['s0_lead3', 's1_textrank', 's2_coverage', 's3_simplified'], selector=SelectorConfig(max_sentences=3, similarity_threshold=0.08, top_entity_k=8, require_number_coverage=True, require_date_coverage=True), simplifier=SimplifyConfig(enabled=True, max_word_length=8, min_word_frequency_rank=2000, max_replacements_per_doc=10, min_replacement_confidence=0.7, max_glossary_items=8, use_glossary_fallback=True, use_structural_simplification=True, protected_words=['said', 'says', 'mr', 'mrs', 'ms', 'government', 'minister', 'president', 'police', 'court', 'judge', 'company', 'officials']))
S4_SELECTOR_CONFIG = {'max_sentences': 4, 'top_entity_k': 8, 'similarity_threshold': 0.08, 'require_nu

## 3. Dataset bootstrap + load selected split

In [6]:
dataset_status = ensure_dataset(
    data_dir=paths.data_dir,
    auto_download=True,
    force_download=False,
)
print(dataset_status)

split_path = paths.data_dir / f"{RUN_SPLIT}.csv"
raw_df, colmap = load_csv(split_path)
df = standardize_dataframe(raw_df, colmap)

# Keep doc_id as the repo-native identifier.
# Also expose an id alias for compatibility if someone downstream expects it.
if "doc_id" not in df.columns:
    raise ValueError("Expected repo-standardized dataframe to contain 'doc_id'.")
df["doc_id"] = df["doc_id"].astype(str)
df["id"] = df["doc_id"]

if RUN_LIMIT is not None:
    df = df.iloc[:RUN_LIMIT].copy()

print("Loaded split:", split_path)
print("Rows:", len(df))
df.head(3)


Found existing local CSV dataset.
Loaded split: /Users/gm/Desktop/4/data/test.csv
Rows: 11490


,doc_id,title,article,reference,id
0,0,,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,0
1,1,,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",1
2,2,,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,2


## 4. Shared I/O helpers

In [7]:
def normalize_pipeline_input(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "doc_id" not in out.columns and "id" in out.columns:
        out["doc_id"] = out["id"]
    required = ["doc_id", "article", "reference"]
    for col in required:
        if col not in out.columns:
            raise ValueError(f"Missing required input column: {col}")
    out["doc_id"] = out["doc_id"].astype(str)
    out["id"] = out["doc_id"]  # compatibility alias
    out["article"] = out["article"].fillna("").astype(str)
    out["reference"] = out["reference"].fillna("").astype(str)
    if "title" not in out.columns:
        out["title"] = ""
    return out

def build_pipeline_output_row(
    *,
    row_id: str,
    system: str,
    article: str,
    reference: str,
    output: str,
    extra: Optional[Dict[str, object]] = None,
) -> Dict[str, object]:
    base = {
        "doc_id": row_id,
        "id": row_id,  # compatibility alias
        "system": system,
        "article": article,
        "reference": reference,
        "output": output,
    }
    if extra:
        base.update(extra)
    return base

def append_shared_metrics(out_df: pd.DataFrame) -> pd.DataFrame:
    metrics_rows = []
    for row in out_df.itertuples(index=False):
        metrics = evaluate_output(source=row.article, reference=row.reference, pred=row.output)
        metrics_rows.append(metrics)
    metrics_df = pd.DataFrame(metrics_rows)
    return pd.concat([out_df.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)


In [8]:
from src.preprocess import (
    token_frequency_rank,
    normalize_whitespace,
    extract_entities,
    extract_numbers,
    extract_dates,
    tokenize,
    split_sentences,
)
from src.evaluation import evaluate_output
from src.simplify import conservative_lexical_simplify
import inspect

print("token_frequency_rank signature:", inspect.signature(token_frequency_rank))
print("normalize_whitespace signature:", inspect.signature(normalize_whitespace))
print("split_sentences signature:", inspect.signature(split_sentences))
print("extract_entities signature:", inspect.signature(extract_entities))
print("conservative_lexical_simplify signature:", inspect.signature(conservative_lexical_simplify))

token_frequency_rank signature: (texts: 'List[str]') -> 'Dict[str, int]'
normalize_whitespace signature: (text: 'str') -> 'str'
split_sentences signature: (text: 'str') -> 'List[str]'
extract_entities signature: (text: 'str') -> 'List[str]'
conservative_lexical_simplify signature: (text: 'str', frequency_rank: 'Dict[str, int]', max_word_length: 'int' = 8, min_word_frequency_rank: 'int' = 2000, max_replacements_per_doc: 'int' = 10, protected_words: 'List[str] | None' = None, use_glossary_fallback: 'bool' = True, min_replacement_confidence: 'float' = 0.7, max_glossary_items: 'int' = 8, use_structural_simplification: 'bool' = True) -> 'Tuple[str, List[Dict[str, str]], List[str]]'


## 4b. Optional validation sweeps for S3 aggressiveness and S4 scaffold selection

In [9]:
from src.preprocess import (
    token_frequency_rank,
    normalize_whitespace,
    extract_entities,
    extract_numbers,
    extract_dates,
    tokenize,
    split_sentences,
)
from src.evaluation import evaluate_output
from src.simplify import conservative_lexical_simplify

def load_named_split_df(split_name: str, limit: Optional[int] = None) -> pd.DataFrame:
    split_path = paths.data_dir / f"{split_name}.csv"
    raw, cmap = load_csv(split_path)
    out = standardize_dataframe(raw, cmap)
    out["doc_id"] = out["doc_id"].astype(str)
    out["id"] = out["doc_id"]
    if limit is not None:
        out = out.iloc[:limit].copy()
    return out

def _shared_outputs_for_df(df_eval: pd.DataFrame, cfg) -> pd.DataFrame:
    work = normalize_pipeline_input(df_eval.copy())
    out = generate_system_outputs(work[["doc_id", "title", "article", "reference"]], cfg).copy()
    out["doc_id"] = out["doc_id"].astype(str)
    return out

def _s3_objective(metrics: Dict[str, float], replacements_count: int) -> float:
    fact_f1 = (metrics.get("entity_f1", 0.0) + metrics.get("number_f1", 0.0) + metrics.get("date_f1", 0.0)) / 3.0
    readability_gain = max(0.0, min(1.0, (12.0 - float(metrics.get("fkgl", 12.0))) / 8.0))
    activity = 1.0 if replacements_count > 0 else 0.0
    precision_penalty = max(
        metrics.get("entity_unsupported_ratio", 0.0),
        metrics.get("number_unsupported_ratio", 0.0),
        metrics.get("date_unsupported_ratio", 0.0),
    )
    return 0.45 * float(metrics.get("rougel", 0.0)) + 0.30 * fact_f1 + 0.20 * readability_gain + 0.05 * activity - 0.10 * precision_penalty

def run_s3_simplifier_sweep(limit: Optional[int] = None) -> pd.DataFrame:
    if limit is None:
        limit = int(SWEEP_LIMIT)
    val_df = load_named_split_df("validation", limit=limit)
    base_cfg = copy.deepcopy(DEFAULT_CONFIG)
    base_cfg.simplifier.enabled = False
    base_shared = _shared_outputs_for_df(val_df, base_cfg)
    s2_rows = base_shared.loc[base_shared["system"] == "s2_coverage", ["doc_id", "article", "reference", "output"]].copy()
    freq_rank = token_frequency_rank(val_df["article"].fillna("").astype(str).tolist())

    grid = [
        {"max_word_length": 8, "min_word_frequency_rank": 2000, "max_replacements_per_doc": 8, "min_replacement_confidence": 0.72, "use_structural_simplification": True},
        {"max_word_length": 8, "min_word_frequency_rank": 2000, "max_replacements_per_doc": 10, "min_replacement_confidence": 0.70, "use_structural_simplification": True},
        {"max_word_length": 8, "min_word_frequency_rank": 1500, "max_replacements_per_doc": 12, "min_replacement_confidence": 0.68, "use_structural_simplification": True},
        {"max_word_length": 9, "min_word_frequency_rank": 2000, "max_replacements_per_doc": 10, "min_replacement_confidence": 0.70, "use_structural_simplification": True},
        {"max_word_length": 8, "min_word_frequency_rank": 2000, "max_replacements_per_doc": 10, "min_replacement_confidence": 0.70, "use_structural_simplification": False},
    ]

    rows = []
    for cfg in grid:
        scores = []
        avg_replacements = []
        for row in s2_rows.itertuples(index=False):
            pred, reps, gloss = conservative_lexical_simplify(
                row.output,
                frequency_rank=freq_rank,
                max_word_length=cfg["max_word_length"],
                min_word_frequency_rank=cfg["min_word_frequency_rank"],
                max_replacements_per_doc=cfg["max_replacements_per_doc"],
                protected_words=DEFAULT_CONFIG.simplifier.protected_words,
                use_glossary_fallback=True,
                min_replacement_confidence=cfg["min_replacement_confidence"],
                max_glossary_items=DEFAULT_CONFIG.simplifier.max_glossary_items,
                use_structural_simplification=cfg["use_structural_simplification"],
            )
            metrics = evaluate_output(source=row.article, reference=row.reference, pred=pred)
            scores.append(_s3_objective(metrics, len(reps)))
            avg_replacements.append(len(reps))
        rows.append({
            **cfg,
            "objective": float(np.mean(scores)) if scores else 0.0,
            "avg_replacements": float(np.mean(avg_replacements)) if avg_replacements else 0.0,
        })

    sweep_df = pd.DataFrame(rows).sort_values(["objective", "avg_replacements"], ascending=[False, False]).reset_index(drop=True)
    return sweep_df

if RUN_S3_SWEEP:
    s3_sweep_results = run_s3_simplifier_sweep(limit=SWEEP_LIMIT)
    display(s3_sweep_results)
    best = s3_sweep_results.iloc[0].to_dict()
    DEFAULT_CONFIG.simplifier.max_word_length = int(best["max_word_length"])
    DEFAULT_CONFIG.simplifier.min_word_frequency_rank = int(best["min_word_frequency_rank"])
    DEFAULT_CONFIG.simplifier.max_replacements_per_doc = int(best["max_replacements_per_doc"])
    DEFAULT_CONFIG.simplifier.min_replacement_confidence = float(best["min_replacement_confidence"])
    DEFAULT_CONFIG.simplifier.use_structural_simplification = bool(best["use_structural_simplification"])
    print("Applied best S3 settings to DEFAULT_CONFIG.simplifier")
else:
    print("RUN_S3_SWEEP = False -> skipping S3 simplifier sweep")

,max_word_length,min_word_frequency_rank,max_replacements_per_doc,min_replacement_confidence,use_structural_simplification,objective,avg_replacements
0,8,2000,10,0.70,False,0.299219,0.163333
1,8,1500,12,0.68,True,0.291824,0.166667
2,8,2000,10,0.70,True,0.291657,0.163333
3,8,2000,8,0.72,True,0.290912,0.146667
4,9,2000,10,0.70,True,0.290346,0.130000


Applied best S3 settings to DEFAULT_CONFIG.simplifier


## 5. Shared S0–S3 using the repo pipeline

In [10]:
SYSTEM_MAP = {
    "s0_lead3": "S0",
    "s1_textrank": "S1",
    "s2_coverage": "S2",
    "s3_simplified": "S3",
}

def run_shared_s0_s3(df_in: pd.DataFrame) -> pd.DataFrame:
    # Use the shared repo pipeline AS-IS and keep its computed metrics.
    work = df_in.copy()
    shared_df = generate_system_outputs(work[["doc_id", "title", "article", "reference"]], DEFAULT_CONFIG).copy()
    shared_df["doc_id"] = shared_df["doc_id"].astype(str)
    shared_df["system"] = shared_df["system"].map(SYSTEM_MAP)

    # Add the teammate-facing `id` alias without removing repo-native `doc_id`.
    insert_at = list(shared_df.columns).index("doc_id") + 1
    shared_df.insert(insert_at, "id", shared_df["doc_id"])
    return shared_df

df_in = normalize_pipeline_input(df)
shared_outputs = run_shared_s0_s3(df_in)

# Reuse the shared S2 outputs directly for S4 instead of recomputing coverage-aware TextRank.
S2_LOOKUP = (
    shared_outputs.loc[shared_outputs["system"] == "S2", ["doc_id", "output"]]
    .drop_duplicates(subset=["doc_id"])
    .set_index("doc_id")["output"]
    .to_dict()
)

print("Shared outputs rows:", len(shared_outputs))
print("Unique docs in S2 lookup:", len(S2_LOOKUP))
shared_outputs.head(8)

Shared outputs rows: 45960
Unique docs in S2 lookup: 11490


,doc_id,id,system,system_label,title,article,reference,output,replacements,glossary,...,entity_unsupported_ratio,number_precision,number_f1,number_unsupported_ratio,date_precision,date_f1,date_unsupported_ratio,novel_token_ratio,copy_token_ratio,novel_content_token_ratio
0,0,0,S0,S0 Lead-3,,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,(CNN)The Palestinian Authority officially beca...,[],[],...,0.0,1.0,0.571429,0.0,1.0,0.666667,0.0,0.0,1.0,0.0
1,0,0,S1,S1 TextRank,,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,"Later that month, the ICC opened a preliminary...",[],[],...,0.0,1.0,0.333333,0.0,1.0,0.666667,0.0,0.0,1.0,0.0
2,0,0,S2,S2 Coverage-aware TextRank,,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinians signed the ICC's founding Rom...,[],[],...,0.0,1.0,0.750000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0
3,0,0,S3,S3 Coverage-aware + simplification,,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinians signed the ICC's founding Rom...,[],[committed = carried out],...,0.0,1.0,0.750000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0
4,1,1,S0,S0 Lead-3,,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",(CNN)Never mind cats having nine lives. A stra...,[],[],...,0.0,0.0,0.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0
5,1,1,S1,S1 TextRank,,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",A stray pooch in Washington State has used up ...,[],[],...,0.0,0.0,0.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0
6,1,1,S2,S2 Coverage-aware TextRank,,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",That's according to Washington State Universit...,[],[],...,0.0,1.0,1.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0
7,1,1,S3,S3 Coverage-aware + simplification,,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",That's according to Washington State Universit...,[],[],...,0.0,1.0,1.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0


## 6. S4 / S5 model helpers

In [11]:
_model_cache = {}

def clear_model_cache():
    global _model_cache
    _model_cache = {}
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def _clean_generation_config(model, tokenizer):
    gen_cfg = copy.deepcopy(model.generation_config)
    # Remove conflicting defaults that spam warnings when max_new_tokens/min_new_tokens are passed.
    if hasattr(gen_cfg, "max_length"):
        gen_cfg.max_length = None
    if hasattr(gen_cfg, "min_length"):
        gen_cfg.min_length = None
    # Some seq2seq checkpoints expect a BOS id in generation config.
    if getattr(gen_cfg, "forced_bos_token_id", None) is None and getattr(tokenizer, "bos_token_id", None) is not None:
        gen_cfg.forced_bos_token_id = tokenizer.bos_token_id
    return gen_cfg

def load_model_bundle(model_name: str):
    if model_name in _model_cache:
        return _model_cache[model_name]
    auth_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    tokenizer = AutoTokenizer.from_pretrained(model_name, **auth_kwargs)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **auth_kwargs)
    if DEVICE != "cpu":
        model = model.to(DEVICE)
    model.eval()
    model.generation_config = _clean_generation_config(model, tokenizer)
    _model_cache[model_name] = (tokenizer, model)
    return _model_cache[model_name]

def generate_with_model(
    model_name: str,
    prompt: str,
    max_new_tokens: int,
    min_new_tokens: int = 0,
    num_beams: int = 4,
    no_repeat_ngram_size: int = 3,
    length_penalty: float = 1.0,
    repetition_penalty: float = 1.05,
) -> str:
    tokenizer, model = load_model_bundle(model_name)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            generation_config=model.generation_config,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            length_penalty=length_penalty,
            repetition_penalty=repetition_penalty,
            do_sample=False,
            early_stopping=True,
            pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return normalize_whitespace(text)

def fact_recall_against_source(source_text: str, candidate_text: str) -> float:
    source_items = set()
    source_items.update([x.lower() for x in extract_entities(source_text)])
    source_items.update([x.lower() for x in extract_numbers(source_text)])
    source_items.update([x.lower() for x in extract_dates(source_text)])
    if not source_items:
        return 1.0
    cand_items = set()
    cand_items.update([x.lower() for x in extract_entities(candidate_text)])
    cand_items.update([x.lower() for x in extract_numbers(candidate_text)])
    cand_items.update([x.lower() for x in extract_dates(candidate_text)])
    return len(source_items & cand_items) / len(source_items)


## 7. S4 = S2 + constrained LLM rewrite

In [12]:
from src.preprocess import extract_entities, extract_numbers, extract_dates, tokenize
from src.evaluation import evaluate_output

def build_s4_prompt(selected_text: str) -> str:
    return (
        "You are simplifying a factual news summary for adult ESL readers.\n"
        "Rewrite the summary in plain English while preserving every factual detail from the source summary.\n"
        "Rules:\n"
        "- Preserve all names, dates, numbers, money amounts, percentages, and places exactly.\n"
        "- Do not add any unsupported facts.\n"
        "- Prefer shorter sentences and common words when meaning stays the same.\n"
        "- Keep the chronology and the main actors the same.\n"
        "- Write 3 to 5 short coherent sentences.\n"
        "- Aim for about 60 to 95 words.\n\n"
        f"Source summary:\n{selected_text}\n\n"
        "Simplified summary:"
    )

def resolve_s4_model_name() -> str:
    ckpt_dir = paths.checkpoint_dir / "s4_finetuned"
    if USE_FINETUNED_S4 and ckpt_dir.exists():
        return str(ckpt_dir)
    return S4_MODEL_NAME

def generate_s4_scaffold(row: pd.Series) -> str:
    row_id = str(row["doc_id"])
    if row_id in S2_LOOKUP and S4_SELECTOR_CONFIG == {
        "max_sentences": DEFAULT_CONFIG.selector.max_sentences,
        "top_entity_k": DEFAULT_CONFIG.selector.top_entity_k,
        "similarity_threshold": DEFAULT_CONFIG.selector.similarity_threshold,
        "require_number_coverage": DEFAULT_CONFIG.selector.require_number_coverage,
        "require_date_coverage": DEFAULT_CONFIG.selector.require_date_coverage,
    }:
        return S2_LOOKUP[row_id]

    from src.selectors import coverage_aware_select as shared_coverage_select
    title = row["title"] if "title" in row else ""
    return shared_coverage_select(
        row["article"],
        title=title,
        max_sentences=S4_SELECTOR_CONFIG["max_sentences"],
        top_entity_k=S4_SELECTOR_CONFIG["top_entity_k"],
        similarity_threshold=S4_SELECTOR_CONFIG["similarity_threshold"],
        require_number_coverage=S4_SELECTOR_CONFIG["require_number_coverage"],
        require_date_coverage=S4_SELECTOR_CONFIG["require_date_coverage"],
    )

def _word_count(text: str) -> int:
    return len(tokenize(text))

def s4_guard_metrics(source_text: str, candidate_text: str) -> Dict[str, float]:
    return evaluate_output(source=source_text, reference="", pred=candidate_text)

def s4_guard_accepts(source_text: str, candidate_text: str, metrics: Optional[Dict[str, float]] = None) -> bool:
    metrics = metrics or s4_guard_metrics(source_text, candidate_text)
    src_entities = extract_entities(source_text)
    src_numbers = extract_numbers(source_text)
    src_dates = extract_dates(source_text)

    entity_ok = (not src_entities) or (
        float(metrics.get("entity_f1", 0.0)) >= S4_MIN_ENTITY_F1
        and float(metrics.get("entity_unsupported_ratio", 1.0)) <= S4_MAX_ENTITY_UNSUPPORTED
    )
    number_ok = (not src_numbers) or (
        float(metrics.get("number_f1", 0.0)) >= S4_MIN_NUMBER_F1
        and float(metrics.get("number_unsupported_ratio", 1.0)) <= S4_MAX_NUMBER_UNSUPPORTED
    )
    date_ok = (not src_dates) or (
        float(metrics.get("date_f1", 0.0)) >= S4_MIN_DATE_F1
        and float(metrics.get("date_unsupported_ratio", 1.0)) <= S4_MAX_DATE_UNSUPPORTED
    )

    wc = _word_count(candidate_text)
    length_ok = S4_MIN_WORDS <= wc <= S4_MAX_WORDS

    return bool(entity_ok and number_ok and date_ok and length_ok)

def run_s4_on_example(row: pd.Series) -> Dict[str, object]:
    scaffold = generate_s4_scaffold(row)
    prompt = build_s4_prompt(scaffold)
    raw = generate_with_model(
        resolve_s4_model_name(),
        prompt,
        max_new_tokens=S4_MAX_NEW_TOKENS,
        min_new_tokens=52,
        num_beams=5,
        no_repeat_ngram_size=3,
        length_penalty=0.95,
        repetition_penalty=1.08,
    )
    guard_metrics = s4_guard_metrics(scaffold, raw)
    accepted = s4_guard_accepts(scaffold, raw, metrics=guard_metrics)

    if not accepted:
        final = scaffold
        fallback_used = True
    else:
        final = raw
        fallback_used = False

    return {
        "output": final,
        "s4_scaffold": scaffold,
        "s4_raw": raw,
        "s4_fallback_used": fallback_used,
        "s4_entity_f1": float(guard_metrics.get("entity_f1", 0.0)),
        "s4_number_f1": float(guard_metrics.get("number_f1", 0.0)),
        "s4_date_f1": float(guard_metrics.get("date_f1", 0.0)),
        "s4_entity_unsupported_ratio": float(guard_metrics.get("entity_unsupported_ratio", 0.0)),
        "s4_number_unsupported_ratio": float(guard_metrics.get("number_unsupported_ratio", 0.0)),
        "s4_date_unsupported_ratio": float(guard_metrics.get("date_unsupported_ratio", 0.0)),
        "s4_word_count": float(guard_metrics.get("word_count", 0.0)),
    }

## 8b. Optional S4 fine-tuning (after tuning, before the final benchmark)

In [13]:
from pathlib import Path
from src.preprocess import normalize_whitespace


def prepare_s4_finetuning_frame(split_name: str, limit: Optional[int] = None) -> pd.DataFrame:
    df_split = load_named_split_df(split_name, limit=limit)
    rows = []
    for row in df_split.itertuples(index=False):
        row_series = pd.Series(row._asdict())
        scaffold = generate_s4_scaffold(row_series)
        rows.append({
            "doc_id": str(row_series["doc_id"]),
            "input_text": build_s4_prompt(scaffold),
            "target_text": normalize_whitespace(row_series["reference"]),
        })
    return pd.DataFrame(rows)

def train_s4_finetuned_model(
    train_limit: Optional[int] = None,
    val_limit: Optional[int] = None,
    model_name: Optional[str] = None,
    epochs: Optional[int] = None,
    batch_size: Optional[int] = None,
) -> Path:
    if train_limit is None:
        train_limit = int(S4_FINETUNE_TRAIN_LIMIT) if S4_FINETUNE_TRAIN_LIMIT is not None else None
    if val_limit is None:
        val_limit = int(S4_FINETUNE_VAL_LIMIT) if S4_FINETUNE_VAL_LIMIT is not None else None
    if model_name is None:
        model_name = str(S4_FINETUNE_MODEL_NAME)
    if epochs is None:
        epochs = int(S4_FINETUNE_EPOCHS)
    if batch_size is None:
        batch_size = int(S4_FINETUNE_BATCH_SIZE)
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        DataCollatorForSeq2Seq,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
    )
    import inspect

    train_df = prepare_s4_finetuning_frame("train", limit=train_limit)
    val_df = prepare_s4_finetuning_frame("validation", limit=val_limit)

    train_ds = Dataset.from_pandas(train_df[["input_text", "target_text"]], preserve_index=False)
    val_ds = Dataset.from_pandas(val_df[["input_text", "target_text"]], preserve_index=False)

    auth_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    tokenizer = AutoTokenizer.from_pretrained(model_name, **auth_kwargs)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **auth_kwargs)

    if hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass

    def preprocess_batch(batch):
        model_inputs = tokenizer(batch["input_text"], max_length=768, truncation=True)
        labels = tokenizer(text_target=batch["target_text"], max_length=160, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names, desc="Tokenizing S4 train")
    val_tok = val_ds.map(preprocess_batch, batched=True, remove_columns=val_ds.column_names, desc="Tokenizing S4 val")

    out_dir = paths.checkpoint_dir / "s4_finetuned"
    out_dir.mkdir(parents=True, exist_ok=True)

    args_sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    args_kwargs = dict(
        output_dir=str(out_dir),
        learning_rate=5e-5,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=2 if DEVICE == "mps" else 1,
        predict_with_generate=False,
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        report_to="none",
        remove_unused_columns=True,
        dataloader_pin_memory=False if DEVICE == "mps" else True,
        fp16=(DEVICE == "cuda"),
    )

    # Transformers changed this kwarg across versions.
    if "evaluation_strategy" in args_sig.parameters:
        args_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in args_sig.parameters:
        args_kwargs["eval_strategy"] = "epoch"
        args_kwargs["do_eval"] = True
    else:
        args_kwargs["do_eval"] = True

    # Keep only kwargs supported by the installed transformers version.
    args_kwargs = {k: v for k, v in args_kwargs.items() if k in args_sig.parameters}

    args = Seq2SeqTrainingArguments(**args_kwargs)

    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    )
    sig = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Seq2SeqTrainer(**trainer_kwargs)
    trainer.train()
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))

    del trainer
    del model
    clear_model_cache()
    return out_dir

if ENABLE_S4_FINETUNING:
    ft_dir = train_s4_finetuned_model()
    USE_FINETUNED_S4 = True
    print("Saved fine-tuned S4 checkpoint to:", ft_dir.resolve())
else:
    print("ENABLE_S4_FINETUNING = False -> skipping S4 fine-tuning")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Tokenizing S4 train:   0%|          | 0/12000 [00:00<?, ? examples/s]

Tokenizing S4 val:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'loss': '5.822', 'grad_norm': '5.307', 'learning_rate': '4.959e-05', 'epoch': '0.008333'}
{'loss': '5.683', 'grad_norm': '4.47', 'learning_rate': '4.918e-05', 'epoch': '0.01667'}
{'loss': '5.897', 'grad_norm': '5.249', 'learning_rate': '4.876e-05', 'epoch': '0.025'}
{'loss': '5.621', 'grad_norm': '5.341', 'learning_rate': '4.834e-05', 'epoch': '0.03333'}
{'loss': '5.643', 'grad_norm': '4.56', 'learning_rate': '4.793e-05', 'epoch': '0.04167'}
{'loss': '5.584', 'grad_norm': '4.667', 'learning_rate': '4.751e-05', 'epoch': '0.05'}
{'loss': '5.634', 'grad_norm': '4.513', 'learning_rate': '4.709e-05', 'epoch': '0.05833'}
{'loss': '5.615', 'grad_norm': '4.599', 'learning_rate': '4.668e-05', 'epoch': '0.06667'}
{'loss': '5.704', 'grad_norm': '5.601', 'learning_rate': '4.626e-05', 'epoch': '0.075'}
{'loss': '5.62', 'grad_norm': '5.089', 'learning_rate': '4.584e-05', 'epoch': '0.08333'}
{'loss': '5.738', 'grad_norm': '4.759', 'learning_rate': '4.542e-05', 'epoch': '0.09167'}
{'loss': '5.578', '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2728', 'train_samples_per_second': '4.398', 'train_steps_per_second': '2.199', 'train_loss': '5.672', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned S4 checkpoint to: /Users/gm/Desktop/4/outputs/checkpoints/s4_finetuned


## 8. S5 = full-article LLM baseline

This cell defines the S5 helper functions. The live S5 progress bar appears later in **Step 9** when `run_s5_system(...)` is actually called.

In [14]:
from typing import Dict, List, Optional
import pandas as pd
from src.preprocess import split_sentences, tokenize

def chunk_article_by_words(article: str, word_budget: Optional[int] = None) -> List[str]:
    if word_budget is None:
        word_budget = int(S5_CHUNK_WORD_BUDGET)

    sents = split_sentences(article)
    chunks, cur, cur_words = [], [], 0

    for sent in sents:
        n = len(tokenize(sent))

        # If a single sentence already exceeds the chunk budget,
        # keep it as its own chunk instead of silently overflowing.
        if n > word_budget:
            if cur:
                chunks.append(" ".join(cur).strip())
                cur, cur_words = [], 0
            chunks.append(sent.strip())
            continue

        if cur and cur_words + n > word_budget:
            chunks.append(" ".join(cur).strip())
            cur, cur_words = [sent], n
        else:
            cur.append(sent)
            cur_words += n

    if cur:
        chunks.append(" ".join(cur).strip())

    return chunks


def build_s5_prompt(article_text: str) -> str:
    return (
        "Summarize this news article in plain English for adult ESL readers.\n"
        "Rules:\n"
        "- Keep the most important names, dates, numbers, and places.\n"
        "- Do not add any new facts.\n"
        "- Write 3 to 5 short, coherent sentences.\n"
        "- Aim for about 60 to 90 words.\n\n"
        f"Article:\n{article_text}\n\n"
        "Summary:"
    )


def build_s5_input(article_text: str) -> str:
    # BART/CNN summarization models usually work better on raw article text
    # than on an instruction wrapper. Keep the prompt for instruction-following models.
    model_name = str(S5_MODEL_NAME).lower()
    if "bart" in model_name:
        return article_text
    return build_s5_prompt(article_text)


def plan_s5_for_article(article: str) -> Dict[str, object]:
    article = str(article)
    chunk_budget = int(S5_CHUNK_WORD_BUDGET)
    if len(tokenize(article)) <= chunk_budget:
        return {"mode": "single_pass", "chunks": None, "steps": 1}

    chunks = chunk_article_by_words(article, word_budget=chunk_budget)
    return {
        "mode": "chunk_then_merge",
        "chunks": chunks,
        "steps": len(chunks) + 1,  # each chunk summary + final merge
    }


def run_s5_on_example(
    row: pd.Series,
    plan: Optional[Dict[str, object]] = None,
    pbar=None,
    completed_steps: int = 0,
) -> tuple[Dict[str, object], int]:
    article = str(row["article"])
    row_id = str(row["doc_id"])

    if plan is None:
        plan = plan_s5_for_article(article)

    if plan["mode"] == "single_pass":
        if pbar is not None:
            pbar.update(completed_steps, last_doc=f"{row_id} | single-pass", force=True)

        out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_input(article),
            max_new_tokens=S5_MAX_NEW_TOKENS,
            min_new_tokens=S5_MIN_NEW_TOKENS,
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            repetition_penalty=1.06,
        ).strip()

        completed_steps += 1
        if pbar is not None:
            pbar.update(completed_steps, last_doc=f"{row_id} | single-pass done", force=True)

        return {
            "output": out,
            "s5_mode": "single_pass",
            "s5_num_chunks": 1,
        }, completed_steps

    chunks = plan["chunks"]
    chunk_summaries = []

    for i, chunk in enumerate(chunks, start=1):
        if pbar is not None:
            pbar.update(completed_steps, last_doc=f"{row_id} | chunk {i}/{len(chunks)}", force=True)

        chunk_out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_input(chunk),
            max_new_tokens=max(72, S5_MAX_NEW_TOKENS // 2),
            min_new_tokens=28,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            repetition_penalty=1.05,
        ).strip()

        chunk_summaries.append(chunk_out)
        completed_steps += 1
        if pbar is not None:
            pbar.update(completed_steps, last_doc=f"{row_id} | finished chunk {i}/{len(chunks)}", force=True)

    merged_source = " ".join(chunk_summaries)

    if pbar is not None:
        pbar.update(completed_steps, last_doc=f"{row_id} | merging", force=True)

    final = generate_with_model(
        S5_MODEL_NAME,
        build_s5_input(merged_source),
        max_new_tokens=S5_MAX_NEW_TOKENS,
        min_new_tokens=S5_MIN_NEW_TOKENS,
        num_beams=5,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        repetition_penalty=1.06,
    ).strip()

    completed_steps += 1
    if pbar is not None:
        pbar.update(completed_steps, last_doc=f"{row_id} | merge done", force=True)

    return {
        "output": final,
        "s5_mode": "chunk_then_merge",
        "s5_num_chunks": len(chunks),
    }, completed_steps

print("S5 helper functions loaded. Progress bar will appear later in Step 9 when S5 is actually run.", flush=True)


S5 helper functions loaded. Progress bar will appear later in Step 9 when S5 is actually run.


## 9. Run S4 / S5 in the shared row format (separate loops for clearer progress and lower memory pressure)

This is the cell that should show the live S4 and S5 progress bars during a normal top-to-bottom run.

In [15]:
def run_s4_system(df_in: pd.DataFrame, show_progress: bool = True) -> pd.DataFrame:
    rows = []
    total = len(df_in)
    print(f"Running S4 on {total} examples...", flush=True)
    pbar = make_progress_bar(total=total, desc="Running S4", enabled=show_progress, update_every=5)
    last_doc = None

    try:
        for idx, row in enumerate(df_in.itertuples(index=False), start=1):
            row_series = pd.Series(row._asdict())
            row_id = str(row_series["doc_id"])
            last_doc = row_id

            s4_payload = run_s4_on_example(row_series)
            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S4",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s4_payload["output"],
                extra={k: v for k, v in s4_payload.items() if k != "output"},
            ))

            if pbar is not None:
                pbar.update(idx, last_doc=row_id)

    finally:
        if pbar is not None:
            pbar.close(last_doc=last_doc)

    out = pd.DataFrame(rows)
    print("Scoring S4 outputs...", flush=True)
    scored = append_shared_metrics(out)
    print("Finished S4.", flush=True)
    return scored


def run_s5_system(df_in: pd.DataFrame, show_progress: bool = True) -> pd.DataFrame:
    rows = []
    plans = []
    total_steps = 0

    print(f"Preparing S5 plan for {len(df_in)} examples...", flush=True)
    for row in df_in.itertuples(index=False):
        row_series = pd.Series(row._asdict())
        plan = plan_s5_for_article(row_series["article"])
        plans.append(plan)
        total_steps += int(plan["steps"])

    print(f"Running S5 on {len(df_in)} examples...", flush=True)
    print(f"Estimated S5 generation steps: {total_steps}", flush=True)
    pbar = make_progress_bar(total=total_steps, desc="Running S5", enabled=show_progress, update_every=1)
    completed_steps = 0
    last_doc = None

    try:
        for row, plan in zip(df_in.itertuples(index=False), plans):
            row_series = pd.Series(row._asdict())
            row_id = str(row_series["doc_id"])
            last_doc = row_id

            s5_payload, completed_steps = run_s5_on_example(
                row_series,
                plan=plan,
                pbar=pbar,
                completed_steps=completed_steps,
            )

            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S5",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s5_payload["output"],
                extra={k: v for k, v in s5_payload.items() if k != "output"},
            ))

    finally:
        if pbar is not None:
            pbar.close(last_doc=last_doc)

    out = pd.DataFrame(rows)
    print("Scoring S5 outputs...", flush=True)
    scored = append_shared_metrics(out)
    print("Finished S5.", flush=True)
    return scored


if ENABLE_LLM_SYSTEMS:
    s4_outputs = run_s4_system(df_in, show_progress=True)

    # Free FLAN-T5 before loading BART.
    clear_model_cache()

    s5_outputs = run_s5_system(df_in, show_progress=True)

    llm_outputs = pd.concat([s4_outputs, s5_outputs], ignore_index=True, sort=False)
else:
    llm_outputs = pd.DataFrame(columns=["doc_id", "id", "system", "article", "reference", "output"])

print("LLM output rows:", len(llm_outputs))
llm_outputs.head(6)

Running S4 on 11490 examples...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Scoring S4 outputs...
Finished S4.
Preparing S5 plan for 11490 examples...
Running S5 on 11490 examples...
Estimated S5 generation steps: 32983


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Scoring S5 outputs...
Finished S5.
LLM output rows: 22980


,doc_id,id,system,article,reference,output,s4_scaffold,s4_raw,s4_fallback_used,s4_entity_f1,...,number_f1,number_unsupported_ratio,date_precision,date_f1,date_unsupported_ratio,novel_token_ratio,copy_token_ratio,novel_content_token_ratio,s5_mode,s5_num_chunks
0,0,0,S4,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinians signed the ICC's founding Rom...,The Palestinians signed the ICC's founding Rom...,"Palestinians signed Rome Statute in January, w...",True,0.235294,...,0.750000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0,NaN,NaN
1,1,1,S4,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",A stray pooch in Washington State has used up ...,A stray pooch in Washington State has used up ...,A stray pooch in Washington State has used up ...,True,0.363636,...,1.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0,NaN,NaN
2,2,2,S4,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,"He is, of course, the Iranian foreign minister...","He is, of course, the Iranian foreign minister...",Iran's foreign minister says he was born in 19...,True,0.625000,...,0.222222,0.0,1.0,0.333333,0.0,0.0,1.0,0.0,NaN,NaN
3,3,3,S4,(CNN)Five Americans who were monitored for thr...,17 Americans were exposed to the Ebola virus w...,(CNN)Five Americans who were monitored for thr...,(CNN)Five Americans who were monitored for thr...,Five Americans released after being exposed to...,True,0.631579,...,0.800000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0,NaN,NaN
4,4,4,S4,(CNN)A Duke student has admitted to hanging a ...,Student is no longer on Duke University campus...,"In a news release, it said the student was no ...","In a news release, it said the student was no ...",NEW: Duke University says student was no longe...,True,0.500000,...,0.666667,0.0,1.0,1.000000,0.0,0.0,1.0,0.0,NaN,NaN
5,5,5,S4,(CNN)He's a blue chip college basketball recru...,College-bound basketball star asks girl with D...,"Trey made the prom-posal (yes, that's what the...","Trey made the prom-posal (yes, that's what the...",Trey made the prom-posal in the gym during Ell...,True,0.428571,...,1.000000,0.0,1.0,1.000000,0.0,0.0,1.0,0.0,NaN,NaN


## 10. Combine all systems and save shared-format outputs

In [16]:
combined = pd.concat([shared_outputs, llm_outputs], ignore_index=True, sort=False)
# keep core columns first
core_cols = ["doc_id", "id", "system", "article", "reference", "output"]
other_cols = [c for c in combined.columns if c not in core_cols]
combined = combined[core_cols + other_cols]

combined_path = paths.output_dir / f"{RUN_TAG}_all_system_outputs.csv"
combined.to_csv(combined_path, index=False)

# Save one CSV per system too
per_system_dir = paths.output_dir / "per_system"
ensure_dirs(per_system_dir)
for sys_name, sub in combined.groupby("system"):
    sub.to_csv(per_system_dir / f"{RUN_TAG}_{sys_name}.csv", index=False)

print("Saved combined outputs to:", combined_path.resolve())
print("Saved per-system CSVs to:", per_system_dir.resolve())
combined.head(12)


Saved combined outputs to: /Users/gm/Desktop/4/outputs/test_s0_s5_v25_no_toggle_all_system_outputs.csv
Saved per-system CSVs to: /Users/gm/Desktop/4/outputs/per_system


,doc_id,id,system,article,reference,output,system_label,title,replacements,glossary,...,s4_fallback_used,s4_entity_f1,s4_number_f1,s4_date_f1,s4_entity_unsupported_ratio,s4_number_unsupported_ratio,s4_date_unsupported_ratio,s4_word_count,s5_mode,s5_num_chunks
0,0,0,S0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,(CNN)The Palestinian Authority officially beca...,S0 Lead-3,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,0,S1,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,"Later that month, the ICC opened a preliminary...",S1 TextRank,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,0,S2,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinians signed the ICC's founding Rom...,S2 Coverage-aware TextRank,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,0,S3,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinians signed the ICC's founding Rom...,S3 Coverage-aware + simplification,,[],[committed = carried out],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1,S0,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",(CNN)Never mind cats having nine lives. A stra...,S0 Lead-3,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,1,S1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",A stray pooch in Washington State has used up ...,S1 TextRank,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,1,S2,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",That's according to Washington State Universit...,S2 Coverage-aware TextRank,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,1,S3,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",That's according to Washington State Universit...,S3 Coverage-aware + simplification,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,2,S0,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,"(CNN)If you've been following the news lately,...",S0 Lead-3,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2,2,S1,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,"He is, of course, the Iranian foreign minister...",S1 TextRank,,[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 11. Aggregate comparison table

In [17]:

metric_cols = [
    "rouge1", "rouge2", "rougel",
    "flesch_reading_ease", "fkgl",
    "entity_coverage", "entity_precision", "entity_f1",
    "number_coverage", "number_precision", "number_f1",
    "date_coverage", "date_precision", "date_f1",
    "entity_unsupported_ratio", "number_unsupported_ratio", "date_unsupported_ratio",
    "copy_token_ratio", "novel_token_ratio", "novel_content_token_ratio",
    "word_count", "sentence_count", "avg_sentence_len",
]
existing_metric_cols = [c for c in metric_cols if c in combined.columns]
summary = combined.groupby("system")[existing_metric_cols].mean().round(4).reset_index()
summary_path = paths.table_dir / f"{RUN_TAG}_metric_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary table to:", summary_path.resolve())
summary


Saved summary table to: /Users/gm/Desktop/4/outputs/tables/test_s0_s5_v25_no_toggle_metric_summary.csv


,system,rouge1,rouge2,rougel,flesch_reading_ease,fkgl,entity_coverage,entity_precision,entity_f1,number_coverage,...,date_f1,entity_unsupported_ratio,number_unsupported_ratio,date_unsupported_ratio,copy_token_ratio,novel_token_ratio,novel_content_token_ratio,word_count,sentence_count,avg_sentence_len
0,S0,0.3799,0.1646,0.2418,48.9074,12.7938,0.1996,0.9993,0.3168,0.2680,...,0.4778,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,76.7527,2.9999,25.5850
1,S1,0.3264,0.1283,0.2171,50.9243,12.0791,0.1928,0.9966,0.3090,0.2105,...,0.4679,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,71.5195,2.9999,23.8406
2,S2,0.3469,0.1411,0.2253,48.6868,12.9105,0.2370,0.9998,0.3681,0.4485,...,0.7920,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,77.7914,2.9999,25.9313
3,S3,0.3466,0.1408,0.2251,48.8312,12.8912,0.2370,0.9997,0.3680,0.4485,...,0.7920,0.0001,0.0000,0.0000,0.9992,0.0008,0.0013,77.8005,2.9999,25.9343
4,S4,0.3381,0.1407,0.2182,49.6659,12.5957,0.2867,0.9998,0.4283,0.5107,...,0.8314,0.0002,0.0000,0.0000,0.9998,0.0002,0.0003,100.6507,3.9893,25.2135
5,S5,0.3925,0.1791,0.2771,56.8183,9.5965,0.1639,0.8954,0.2652,0.2881,...,0.5011,0.1042,0.0086,0.0017,0.9882,0.0118,0.0168,56.2183,3.4526,17.1566


## 12. One-example preview across all systems

In [18]:
example_id = str(df_in.iloc[0]["doc_id"])
example_rows = combined[combined["doc_id"] == example_id][["system", "output"]].copy()
example_rows


,system,output
0,S0,(CNN)The Palestinian Authority officially beca...
1,S1,"Later that month, the ICC opened a preliminary..."
2,S2,The Palestinians signed the ICC's founding Rom...
3,S3,The Palestinians signed the ICC's founding Rom...
45960,S4,The Palestinians signed the ICC's founding Rom...
57450,S5,Palestinian Authority becomes 123rd member of ...
